# Oil-VWRA Trading Signal Analysis

**Thesis**: Use Brent crude oil price movements as a leading indicator for VWRA (Vanguard FTSE All-World UCITS ETF) trading signals, exploiting the war-driven macro regime where oil shocks drive global equity sentiment.

**Key assumptions**:
- This is a **temporary macro shock regime**, NOT a stable pair — no cointegration assumed
- Human-speed execution on IBKR (signals lasting minutes to hours)
- Brent is primary oil input; WTI as secondary confirmation
- Controls: ES/NQ futures, DXY, Gold
- Trading windows: London open + headline-active hours
- Cost model: IBKR commissions + half-spread + 1 tick slippage

**Algos implemented**:
1. Regime-Switching Oil-Shock Model (3-state: normal / war risk-off / de-escalation relief)
2. London-Open Oil Gap Model (continuation vs fade)
3. Dynamic-Beta / Kalman Filter Lead-Lag Model
4. Cross-Asset Confirmation Momentum
5. Failed Oil-Spike Reversal
6. Fair-Value Residual / Z-Score Model

In [ ]:
# ============================================================
# Cell 1: Imports & Configuration
# ============================================================
import json
import urllib.request
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats, signal
from scipy.optimize import minimize

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style="darkgrid", palette="husl")
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['figure.dpi'] = 100

# ============================================================
# Configuration
# ============================================================
SYMBOLS = {
    'VWRA':  'VWRA.L',     # Vanguard FTSE All-World (LSE)
    'Brent': 'BZ=F',       # Brent Crude Futures
    'WTI':   'CL=F',       # WTI Crude Futures
    'ES':    'ES=F',        # S&P 500 E-mini Futures
    'NQ':    'NQ=F',        # Nasdaq 100 E-mini Futures
    'DXY':   'DX-Y.NYB',   # US Dollar Index
    'Gold':  'GC=F',        # Gold Futures
}

# IBKR cost model for VWRA.L
COST_MODEL = {
    'commission_per_share': 0.05,    # GBP per share (IBKR tiered)
    'half_spread_bps': 5,            # ~5 bps half-spread for VWRA
    'slippage_ticks': 1,             # 1 tick = 0.02 GBP for VWRA
    'tick_size': 0.02,
    'min_commission': 1.25,          # GBP minimum
}

# Signal thresholds
SIGNAL_HORIZON_HOURS = [1, 4]  # Forward return horizons for backtesting
LOOKBACK_START = '2025-03-01'

print("Configuration loaded.")

## 1. Data Acquisition

Fetch hourly and 5-minute data from Yahoo Finance for all instruments since March 1, 2025.

In [ ]:
# ============================================================
# Cell 2: Data Fetching
# ============================================================
def fetch_yahoo(symbol, range_val='1mo', interval='1h'):
    """Fetch OHLCV data from Yahoo Finance v8 API."""
    url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
           f"?range={range_val}&interval={interval}&includePrePost=true")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (DegenTrader/1.0)"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read())
    
    result = data["chart"]["result"][0]
    meta = result["meta"]
    timestamps = result["timestamp"]
    quote = result["indicators"]["quote"][0]
    
    df = pd.DataFrame({
        'open':   quote['open'],
        'high':   quote['high'],
        'low':    quote['low'],
        'close':  quote['close'],
        'volume': quote['volume'],
    }, index=pd.to_datetime(timestamps, unit='s', utc=True))
    
    df.index.name = 'datetime'
    df = df.dropna(subset=['close'])
    
    # Add returns
    df['ret'] = df['close'].pct_change()
    df['log_ret'] = np.log(df['close'] / df['close'].shift(1))
    
    return df, meta

# Fetch hourly data (1 month — max available at hourly resolution)
print("Fetching hourly data...")
hourly_data = {}
for name, sym in SYMBOLS.items():
    try:
        df, meta = fetch_yahoo(sym, range_val='1mo', interval='1h')
        hourly_data[name] = df
        print(f"  {name:6s} ({sym:10s}): {len(df):4d} bars, "
              f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}, "
              f"last={df['close'].iloc[-1]:.2f}")
    except Exception as e:
        print(f"  {name:6s}: FAILED — {e}")

# Fetch 5-minute data (last 60 days max, but Yahoo gives ~7-8 days)
print("\nFetching 5-minute data...")
fivemin_data = {}
for name, sym in SYMBOLS.items():
    try:
        df, meta = fetch_yahoo(sym, range_val='1mo', interval='5m')
        fivemin_data[name] = df
        print(f"  {name:6s} ({sym:10s}): {len(df):5d} bars, "
              f"{df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
    except Exception as e:
        print(f"  {name:6s}: FAILED — {e}")

print("\nData acquisition complete.")

In [ ]:
# ============================================================
# Cell 3: Build aligned returns matrix (hourly)
# ============================================================
# Merge all instruments into a single DataFrame on common timestamps
# Oil trades nearly 24h; VWRA only during LSE hours (8:00-16:30 UK)
# We keep all timestamps and forward-fill VWRA during gaps

close_prices = pd.DataFrame({
    name: df['close'] for name, df in hourly_data.items()
})
close_prices = close_prices.sort_index()

# Forward-fill VWRA for up to 18 hours (overnight gap)
close_prices['VWRA'] = close_prices['VWRA'].ffill(limit=18)

# Drop rows where Brent is NaN (market fully closed)
close_prices = close_prices.dropna(subset=['Brent'])

# Compute returns
returns = close_prices.pct_change().dropna()

# Brent-WTI spread
close_prices['Brent_WTI_spread'] = close_prices['Brent'] - close_prices['WTI']

# Realized vol (rolling 24-hour)
for asset in ['Brent', 'VWRA']:
    returns[f'{asset}_rvol_24h'] = returns[asset].rolling(24).std() * np.sqrt(24)

print(f"Aligned price matrix: {close_prices.shape}")
print(f"Aligned returns matrix: {returns.shape}")
print(f"\nDate range: {returns.index[0]} to {returns.index[-1]}")
print(f"\nLatest prices:")
display(close_prices.iloc[-1].to_frame('Last Price').T)
print(f"\nReturn correlations (full sample):")
display(returns[['VWRA', 'Brent', 'WTI', 'ES', 'NQ', 'DXY', 'Gold']].corr().round(3))

## 2. Correlation & Lead-Lag Analysis

Understanding the time-varying relationship between Brent and VWRA, including which leads which and by how much.

In [ ]:
# ============================================================
# Cell 4: Lead-Lag Cross-Correlation Analysis
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

# 1. Cross-correlation: Brent returns vs VWRA returns at various lags
max_lag = 12  # hours
lags = range(-max_lag, max_lag + 1)
brent_ret = returns['Brent'].dropna()
vwra_ret = returns['VWRA'].dropna()
common_idx = brent_ret.index.intersection(vwra_ret.index)
brent_aligned = brent_ret.loc[common_idx]
vwra_aligned = vwra_ret.loc[common_idx]

xcorr = [brent_aligned.corr(vwra_aligned.shift(-lag)) for lag in lags]

ax = axes[0, 0]
colors = ['#e74c3c' if lag < 0 else '#2ecc71' if lag > 0 else '#3498db' for lag in lags]
ax.bar(lags, xcorr, color=colors, alpha=0.8, edgecolor='white')
ax.axhline(y=0, color='white', linewidth=0.5)
ax.axvline(x=0, color='yellow', linewidth=1, linestyle='--', alpha=0.7)
ax.set_xlabel('Lag (hours) — positive means Brent leads VWRA')
ax.set_ylabel('Correlation')
ax.set_title('Cross-Correlation: Brent Returns → VWRA Returns\n(positive lag = Brent LEADS VWRA)')
peak_lag = lags[np.argmax(np.abs(xcorr))]
ax.annotate(f'Peak at lag={peak_lag}h\nr={max(xcorr, key=abs):.3f}',
            xy=(peak_lag, xcorr[list(lags).index(peak_lag)]),
            fontsize=11, fontweight='bold', color='yellow',
            xytext=(peak_lag + 2, max(xcorr) * 0.8),
            arrowprops=dict(arrowstyle='->', color='yellow'))

# 2. Rolling 48-hour correlation
ax = axes[0, 1]
rolling_corr = brent_aligned.rolling(48).corr(vwra_aligned)
ax.plot(rolling_corr.index, rolling_corr.values, color='#e67e22', linewidth=1.5, alpha=0.8)
ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
ax.axhline(y=rolling_corr.mean(), color='cyan', linewidth=1, linestyle=':', label=f'Mean={rolling_corr.mean():.3f}')
ax.fill_between(rolling_corr.index, rolling_corr.values, 0, alpha=0.3,
                where=rolling_corr > 0, color='green', label='Positive corr')
ax.fill_between(rolling_corr.index, rolling_corr.values, 0, alpha=0.3,
                where=rolling_corr <= 0, color='red', label='Negative corr')
ax.set_title('Rolling 48-Hour Correlation: Brent vs VWRA Returns')
ax.set_ylabel('Correlation')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# 3. Cross-correlation with controls (ES, DXY)
ax = axes[1, 0]
pairs = [('Brent', 'VWRA'), ('ES', 'VWRA'), ('DXY', 'VWRA'), ('Gold', 'VWRA')]
for leader, follower in pairs:
    ret_l = returns[leader].dropna()
    ret_f = returns[follower].dropna()
    ci = ret_l.index.intersection(ret_f.index)
    xc = [ret_l.loc[ci].corr(ret_f.loc[ci].shift(-lag)) for lag in lags]
    ax.plot(lags, xc, label=f'{leader} → {follower}', linewidth=2, alpha=0.8)
ax.axhline(y=0, color='white', linewidth=0.5)
ax.axvline(x=0, color='yellow', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('Correlation')
ax.set_title('Cross-Correlation: Various Assets → VWRA')
ax.legend(fontsize=9)

# 4. Brent-WTI spread over time
ax = axes[1, 1]
spread = close_prices['Brent_WTI_spread'].dropna()
ax.plot(spread.index, spread.values, color='#9b59b6', linewidth=1.5)
ax.fill_between(spread.index, spread.values, spread.mean(), alpha=0.3, color='#9b59b6')
ax.axhline(y=spread.mean(), color='cyan', linestyle=':', label=f'Mean={spread.mean():.2f}')
ax.set_title('Brent-WTI Spread ($/barrel)')
ax.set_ylabel('Spread ($)')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.suptitle('Lead-Lag & Correlation Analysis — March 2025', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Print key findings
print(f"\n{'='*60}")
print("KEY FINDINGS — Lead-Lag Analysis")
print(f"{'='*60}")
print(f"Peak cross-correlation lag: {peak_lag} hours (Brent leads VWRA)")
print(f"Peak correlation value: {max(xcorr, key=abs):.4f}")
print(f"Contemporaneous correlation: {xcorr[list(lags).index(0)]:.4f}")
print(f"Rolling 48h corr — Mean: {rolling_corr.mean():.4f}, Std: {rolling_corr.std():.4f}")
print(f"Rolling 48h corr — % Positive: {(rolling_corr > 0).mean()*100:.1f}%")
print(f"Brent-WTI spread — Mean: ${spread.mean():.2f}, Current: ${spread.iloc[-1]:.2f}")

## 3. Backtest Engine & Cost Model

Common infrastructure used by all algos: transaction cost model, signal evaluation, and P&L tracking.

In [ ]:
# ============================================================
# Cell 5: Backtest Engine
# ============================================================

def compute_transaction_cost(price, shares=10):
    """IBKR cost model for VWRA.L: commission + half-spread + slippage."""
    commission = max(COST_MODEL['min_commission'], shares * COST_MODEL['commission_per_share'])
    half_spread = price * COST_MODEL['half_spread_bps'] / 10000
    slippage = COST_MODEL['slippage_ticks'] * COST_MODEL['tick_size']
    cost_per_share = half_spread + slippage
    total = commission + shares * cost_per_share
    return total / (shares * price)  # as fraction of notional

def backtest_signals(signals_df, prices, horizon_hours=1, cost_bps=None):
    """
    Backtest a signal series against forward VWRA returns.
    
    signals_df: DataFrame with 'signal' column (-1=sell, 0=flat, 1=buy) and 'strength' (0-1)
    prices: VWRA close prices aligned to signals
    horizon_hours: holding period
    cost_bps: override cost in bps (otherwise uses IBKR model)
    
    Returns DataFrame with signal, forward return, P&L, cumulative P&L.
    """
    if cost_bps is None:
        avg_price = prices.mean()
        cost_bps = compute_transaction_cost(avg_price) * 10000  # convert to bps
    
    cost_frac = cost_bps / 10000
    
    # Forward returns
    fwd_ret = prices.pct_change(horizon_hours).shift(-horizon_hours)
    
    result = pd.DataFrame({
        'signal': signals_df['signal'],
        'strength': signals_df.get('strength', 1.0),
        'price': prices,
        'fwd_return': fwd_ret,
    }, index=signals_df.index)
    
    # P&L = signal * forward_return - |signal_change| * cost
    result['gross_pnl'] = result['signal'] * result['fwd_return']
    result['signal_change'] = result['signal'].diff().abs().clip(0, 2)
    result['cost'] = result['signal_change'] * cost_frac
    result['net_pnl'] = result['gross_pnl'] - result['cost']
    result['cum_pnl'] = result['net_pnl'].cumsum()
    result['cum_gross'] = result['gross_pnl'].cumsum()
    
    return result.dropna(subset=['fwd_return'])

def summarize_backtest(bt, name="Strategy"):
    """Print summary statistics for a backtest."""
    trades = bt[bt['signal'] != 0]
    wins = trades[trades['net_pnl'] > 0]
    
    total_ret = bt['cum_pnl'].iloc[-1] if len(bt) > 0 else 0
    n_trades = (bt['signal_change'] > 0).sum()
    win_rate = len(wins) / len(trades) * 100 if len(trades) > 0 else 0
    avg_win = wins['net_pnl'].mean() * 10000 if len(wins) > 0 else 0
    avg_loss = trades[trades['net_pnl'] <= 0]['net_pnl'].mean() * 10000 if len(trades) > len(wins) else 0
    sharpe = trades['net_pnl'].mean() / trades['net_pnl'].std() * np.sqrt(252 * 8) if trades['net_pnl'].std() > 0 else 0
    max_dd = (bt['cum_pnl'] - bt['cum_pnl'].cummax()).min() * 10000
    total_cost = bt['cost'].sum() * 10000
    
    stats = {
        'Strategy': name,
        'Total Return (bps)': f"{total_ret * 10000:.1f}",
        'Gross Return (bps)': f"{bt['cum_gross'].iloc[-1] * 10000:.1f}" if len(bt) > 0 else "0",
        'Total Cost (bps)': f"{total_cost:.1f}",
        'N Trades': int(n_trades),
        'Win Rate %': f"{win_rate:.1f}",
        'Avg Win (bps)': f"{avg_win:.2f}",
        'Avg Loss (bps)': f"{avg_loss:.2f}",
        'Sharpe (ann.)': f"{sharpe:.2f}",
        'Max DD (bps)': f"{max_dd:.1f}",
        'Bars in Market %': f"{(bt['signal'] != 0).mean() * 100:.1f}",
    }
    return stats

print(f"Avg transaction cost estimate: {compute_transaction_cost(165, 10)*10000:.1f} bps round-trip")
print("Backtest engine ready.")

## 4. Algorithm 1: Regime-Switching Oil-Shock Model

3-state model: **Normal** / **War Risk-Off** / **De-escalation Relief**

Inputs: Brent returns, Brent realized vol, Brent-WTI spread, ES/NQ, DXY, Gold.
Only trade when regime probability > 70%.

In [ ]:
# ============================================================
# Cell 6: Algo 1 — Regime-Switching Oil-Shock Model
# ============================================================
# 3-state HMM-like model using observable features to classify regime:
#   State 0: NORMAL — low oil vol, stable spread, no clear signal → flat
#   State 1: WAR RISK-OFF — oil spiking, high vol, wide spread, equities down → SELL VWRA
#   State 2: DE-ESCALATION RELIEF — oil dropping, vol cooling, spread narrowing → BUY VWRA
#
# We use a simple scoring approach (no external HMM library needed):
# Score each feature, combine, assign state via thresholds.

def algo1_regime_switching(returns_df, prices_df, lookback=24, threshold=0.70):
    """
    Regime-switching oil-shock model.
    Returns DataFrame with 'signal', 'strength', 'regime', 'regime_prob'.
    """
    df = pd.DataFrame(index=returns_df.index)
    
    # Feature 1: Brent short-term momentum (4h return)
    df['brent_mom_4h'] = prices_df['Brent'].pct_change(4)
    
    # Feature 2: Brent realized vol (rolling 24h) z-score
    brent_rvol = returns_df['Brent'].rolling(lookback).std() * np.sqrt(lookback)
    df['brent_vol_z'] = (brent_rvol - brent_rvol.rolling(72).mean()) / brent_rvol.rolling(72).std()
    
    # Feature 3: Brent-WTI spread z-score (widening = war premium)
    bwt_spread = prices_df['Brent'] - prices_df['WTI']
    df['spread_z'] = (bwt_spread - bwt_spread.rolling(72).mean()) / bwt_spread.rolling(72).std()
    
    # Feature 4: ES/NQ weakness (negative = risk-off)
    df['equity_mom'] = (prices_df['ES'].pct_change(4) + prices_df['NQ'].pct_change(4)) / 2
    
    # Feature 5: DXY strength (positive = risk-off / flight to safety)
    df['dxy_mom'] = prices_df['DXY'].pct_change(4)
    
    # Feature 6: Gold strength (positive = risk-off)
    df['gold_mom'] = prices_df['Gold'].pct_change(4)
    
    # Score: positive = risk-off (sell VWRA), negative = relief (buy VWRA)
    # Oil up + vol up + spread wide + equities down + DXY up + gold up = WAR RISK-OFF
    df['risk_off_score'] = (
        np.sign(df['brent_mom_4h']) * 2.0 +          # oil up = risk-off
        df['brent_vol_z'].clip(-2, 2) * 1.0 +         # high vol = risk-off
        df['spread_z'].clip(-2, 2) * 0.8 +             # wide spread = risk-off  
        -np.sign(df['equity_mom']) * 1.5 +              # equities down = risk-off
        np.sign(df['dxy_mom']) * 0.7 +                  # DXY up = risk-off
        np.sign(df['gold_mom']) * 0.5                   # gold up = risk-off
    )
    
    # Normalize to [-1, 1] range
    max_possible = 2.0 + 2.0 + 1.6 + 1.5 + 0.7 + 0.5  # = 6.3
    df['norm_score'] = df['risk_off_score'] / max_possible
    
    # Regime classification with probabilities
    # Smooth the score to avoid whipsawing
    df['smooth_score'] = df['norm_score'].rolling(3).mean()
    
    # Map to regime probabilities using sigmoid-like function
    df['war_prob'] = 1 / (1 + np.exp(-5 * (df['smooth_score'] - 0.3)))
    df['relief_prob'] = 1 / (1 + np.exp(-5 * (-df['smooth_score'] - 0.3)))
    df['normal_prob'] = 1 - df['war_prob'] - df['relief_prob']
    df['normal_prob'] = df['normal_prob'].clip(0, 1)
    
    # Assign regime
    df['regime'] = 'NORMAL'
    df.loc[df['war_prob'] > threshold, 'regime'] = 'WAR_RISK_OFF'
    df.loc[df['relief_prob'] > threshold, 'regime'] = 'RELIEF'
    
    # Signal: sell in war risk-off, buy in relief, flat in normal
    df['signal'] = 0
    df.loc[df['regime'] == 'WAR_RISK_OFF', 'signal'] = -1
    df.loc[df['regime'] == 'RELIEF', 'signal'] = 1
    
    # Strength = max regime probability
    df['strength'] = df[['war_prob', 'relief_prob']].max(axis=1)
    df['regime_prob'] = df['strength']
    
    return df.dropna()

# Run Algo 1
algo1_signals = algo1_regime_switching(returns, close_prices)

# Visualize regimes
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [2, 1, 1]})

# Price chart with regime overlay
ax = axes[0]
vwra_plot = close_prices['VWRA'].dropna()
ax.plot(vwra_plot.index, vwra_plot.values, color='white', linewidth=1.5, label='VWRA', zorder=3)

# Color background by regime
for regime, color in [('WAR_RISK_OFF', '#e74c3c'), ('RELIEF', '#2ecc71')]:
    mask = algo1_signals['regime'] == regime
    regime_periods = algo1_signals[mask].index
    for ts in regime_periods:
        ax.axvspan(ts, ts + pd.Timedelta(hours=1), alpha=0.15, color=color, linewidth=0)

ax.set_title('Algo 1: Regime-Switching Oil-Shock Model — VWRA Price with Regime Overlay', fontsize=13)
ax.set_ylabel('VWRA Price (GBP)')
ax.legend(loc='upper left')

# Red/green patches for legend
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', alpha=0.3, label='WAR RISK-OFF (Sell)'),
    mpatches.Patch(color='#2ecc71', alpha=0.3, label='RELIEF (Buy)'),
    plt.Line2D([0], [0], color='white', linewidth=1.5, label='VWRA'),
], loc='upper left')

# Regime probabilities
ax = axes[1]
ax.fill_between(algo1_signals.index, algo1_signals['war_prob'], alpha=0.6, color='#e74c3c', label='War Risk-Off Prob')
ax.fill_between(algo1_signals.index, -algo1_signals['relief_prob'], alpha=0.6, color='#2ecc71', label='Relief Prob')
ax.axhline(y=0.7, color='red', linestyle=':', alpha=0.5, label='Threshold (70%)')
ax.axhline(y=-0.7, color='green', linestyle=':', alpha=0.5)
ax.set_ylabel('Regime Probability')
ax.set_title('Regime Probabilities')
ax.legend(fontsize=9)

# Signal output
ax = axes[2]
colors = algo1_signals['signal'].map({-1: '#e74c3c', 0: '#95a5a6', 1: '#2ecc71'})
ax.bar(algo1_signals.index, algo1_signals['signal'], color=colors, alpha=0.7, width=0.04)
ax.set_ylabel('Signal (-1=Sell, 0=Flat, +1=Buy)')
ax.set_title('Trading Signal')
ax.set_ylim(-1.5, 1.5)

for a in axes:
    a.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

# Regime statistics
regime_counts = algo1_signals['regime'].value_counts()
print(f"\nRegime Distribution:")
for r, c in regime_counts.items():
    pct = c / len(algo1_signals) * 100
    print(f"  {r:15s}: {c:4d} bars ({pct:.1f}%)")
print(f"\nCurrent Regime: {algo1_signals['regime'].iloc[-1]}")
print(f"Current Signal: {['SELL','FLAT','BUY'][algo1_signals['signal'].iloc[-1]+1]}")
print(f"War Risk-Off Prob: {algo1_signals['war_prob'].iloc[-1]:.1%}")
print(f"Relief Prob: {algo1_signals['relief_prob'].iloc[-1]:.1%}")

## 5. Algorithm 2: London-Open Oil Gap Model

VWRA trades only during LSE hours (08:00–16:30 UK). Oil trades overnight. This algo exploits the **catch-up gap** at London open.

- **Continuation**: Large overnight Brent move, VWRA gaps partially, Brent still extending → trade in direction of gap
- **Fade**: VWRA gaps hard but Brent stalls/reverses in first minutes → fade the gap

In [ ]:
# ============================================================
# Cell 7: Algo 2 — London-Open Oil Gap Model
# ============================================================
# Key insight: oil trades while VWRA is closed.
# At London open (8am UK), VWRA "catches up" to overnight oil moves.
# This creates a predictable trading window.

def algo2_london_gap(hourly_data_dict, close_prices_df):
    """
    London-open oil gap model.
    For each trading day, compute:
      - Overnight Brent return (from LSE close 16:30 to next day 8:00)
      - VWRA opening gap
      - Brent's first-hour direction after LSE open
    Then generate continuation or fade signals.
    """
    brent = hourly_data_dict['Brent']
    vwra = hourly_data_dict['VWRA']
    
    # Work in UK time (UTC for now — LSE hours are 8:00-16:30 UTC in GMT)
    signals_list = []
    
    # Get unique dates from VWRA data
    vwra_dates = vwra.index.normalize().unique()
    
    for i in range(1, len(vwra_dates)):
        today = vwra_dates[i]
        yesterday = vwra_dates[i - 1]
        
        # Yesterday's LSE close (around 16:00 UTC)
        yest_mask = (brent.index >= yesterday) & (brent.index < yesterday + pd.Timedelta(hours=17))
        if len(brent.loc[yest_mask]) == 0:
            continue
        brent_yest_close = brent.loc[yest_mask, 'close'].iloc[-1]
        
        # Today's LSE open area (8:00-9:00 UTC)
        open_start = today + pd.Timedelta(hours=8)
        open_end = today + pd.Timedelta(hours=9)
        
        brent_open_mask = (brent.index >= open_start) & (brent.index <= open_end)
        vwra_open_mask = (vwra.index >= open_start) & (vwra.index <= open_end)
        
        if len(brent.loc[brent_open_mask]) == 0 or len(vwra.loc[vwra_open_mask]) == 0:
            continue
        
        brent_at_open = brent.loc[brent_open_mask, 'close'].iloc[0]
        vwra_prev_close_mask = (vwra.index >= yesterday) & (vwra.index < today)
        if len(vwra.loc[vwra_prev_close_mask]) == 0:
            continue
        vwra_prev_close = vwra.loc[vwra_prev_close_mask, 'close'].iloc[-1]
        vwra_at_open = vwra.loc[vwra_open_mask, 'close'].iloc[0]
        
        # Overnight Brent return
        overnight_brent_ret = (brent_at_open - brent_yest_close) / brent_yest_close
        
        # VWRA opening gap
        vwra_gap = (vwra_at_open - vwra_prev_close) / vwra_prev_close
        
        # Brent first-hour direction (from 8:00 to 9:00)
        if len(brent.loc[brent_open_mask]) >= 2:
            brent_first_hour_ret = (brent.loc[brent_open_mask, 'close'].iloc[-1] - 
                                     brent.loc[brent_open_mask, 'close'].iloc[0]) / brent.loc[brent_open_mask, 'close'].iloc[0]
        else:
            brent_first_hour_ret = 0
        
        # Gap ratio: how much of oil move did VWRA absorb?
        gap_ratio = vwra_gap / overnight_brent_ret if abs(overnight_brent_ret) > 0.001 else 0
        
        # Signal logic
        signal = 0
        variant = 'NONE'
        strength = 0.0
        
        # Only trade if overnight Brent move is significant (> 0.5%)
        if abs(overnight_brent_ret) > 0.005:
            if abs(gap_ratio) < 0.5 and np.sign(brent_first_hour_ret) == np.sign(overnight_brent_ret):
                # CONTINUATION: VWRA hasn't caught up, Brent still extending
                # If oil went UP overnight → risk-off → SELL VWRA
                # If oil went DOWN overnight → relief → BUY VWRA
                signal = -np.sign(overnight_brent_ret)
                variant = 'CONTINUATION'
                strength = min(abs(overnight_brent_ret) * 50, 1.0)  # Scale by move size
                
            elif abs(gap_ratio) > 1.5 and np.sign(brent_first_hour_ret) != np.sign(overnight_brent_ret):
                # FADE: VWRA over-gapped, Brent reversing
                signal = np.sign(vwra_gap)  # Fade = trade against the gap
                variant = 'FADE'
                strength = min(abs(gap_ratio - 1.0) * 0.5, 1.0)
        
        # Generate signals for the morning trading window (8:00-12:00)
        morning_mask = (close_prices_df.index >= open_start) & (close_prices_df.index < today + pd.Timedelta(hours=12))
        for ts in close_prices_df.index[morning_mask]:
            signals_list.append({
                'datetime': ts,
                'signal': int(signal),
                'strength': strength,
                'variant': variant,
                'overnight_brent_ret': overnight_brent_ret,
                'vwra_gap': vwra_gap,
                'gap_ratio': gap_ratio,
                'brent_first_hour': brent_first_hour_ret,
            })
    
    result = pd.DataFrame(signals_list)
    if len(result) == 0:
        return pd.DataFrame(columns=['signal', 'strength', 'variant'], index=close_prices_df.index).fillna(0)
    result = result.set_index('datetime')
    
    # Reindex to full timeline, fill non-gap hours with 0 (flat)
    full = pd.DataFrame(index=close_prices_df.index)
    full = full.join(result)
    full['signal'] = full['signal'].fillna(0).astype(int)
    full['strength'] = full['strength'].fillna(0)
    full['variant'] = full['variant'].fillna('NONE')
    
    return full

algo2_signals = algo2_london_gap(hourly_data, close_prices)

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Gap analysis scatter
gap_data = algo2_signals[algo2_signals['variant'] != 'NONE'].copy()
if len(gap_data) > 0:
    ax = axes[0]
    colors_map = {'CONTINUATION': '#3498db', 'FADE': '#e67e22'}
    for variant in ['CONTINUATION', 'FADE']:
        mask = gap_data['variant'] == variant
        if mask.any():
            scatter = ax.scatter(gap_data.loc[mask, 'overnight_brent_ret'] * 100, 
                               gap_data.loc[mask, 'vwra_gap'] * 100,
                               c=colors_map[variant], label=variant, s=80, alpha=0.7, edgecolors='white')
    ax.axhline(y=0, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='white', linewidth=0.5, alpha=0.5)
    # Add y=x line (perfect gap)
    lim = max(abs(ax.get_xlim()[0]), abs(ax.get_xlim()[1]), abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
    ax.plot([-lim, lim], [-lim, lim], 'w--', alpha=0.3, label='Perfect gap (1:1)')
    ax.set_xlabel('Overnight Brent Return (%)')
    ax.set_ylabel('VWRA Opening Gap (%)')
    ax.set_title('Algo 2: London-Open Gap — Overnight Brent Move vs VWRA Gap')
    ax.legend()

# Signals over time
ax = axes[1]
sig_colors = algo2_signals['signal'].map({-1: '#e74c3c', 0: '#95a5a6', 1: '#2ecc71'})
ax.bar(algo2_signals.index, algo2_signals['signal'], color=sig_colors, alpha=0.7, width=0.04)
ax.set_ylabel('Signal')
ax.set_title('London Gap Trading Signals')
ax.set_ylim(-1.5, 1.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

# Stats
active = algo2_signals[algo2_signals['signal'] != 0]
print(f"\nAlgo 2 — London Gap Model:")
print(f"  Total signal bars: {len(active)} / {len(algo2_signals)} ({len(active)/len(algo2_signals)*100:.1f}%)")
if len(gap_data) > 0:
    print(f"  Continuation signals: {(gap_data['variant']=='CONTINUATION').sum()}")
    print(f"  Fade signals: {(gap_data['variant']=='FADE').sum()}")
    print(f"  Avg overnight Brent move (when signal): {gap_data['overnight_brent_ret'].abs().mean()*100:.2f}%")
print(f"  Current signal: {['SELL','FLAT','BUY'][algo2_signals['signal'].iloc[-1]+1]}")

## 6. Algorithm 3: Dynamic-Beta / Kalman Filter Lead-Lag Model

Estimates time-varying beta: `r_VWRA ≈ α_t + β1_t * r_Brent(lag) + β2_t * r_ES(lag) + β3_t * r_DXY(lag)`

Trade when predicted move exceeds estimated costs by a decent margin.

In [ ]:
# ============================================================
# Cell 8: Algo 3 — Dynamic-Beta / Kalman Filter Lead-Lag
# ============================================================
# Simple Kalman filter to track time-varying betas between
# VWRA and lagged Brent, ES, DXY returns.

def algo3_kalman_beta(returns_df, prices_df, lag=1, 
                       process_noise=1e-4, obs_noise=1e-3, 
                       cost_threshold_bps=15):
    """
    Kalman filter dynamic-beta model.
    State: [alpha, beta_brent, beta_es, beta_dxy]
    Observation: r_VWRA_t = alpha + beta_brent * r_Brent_{t-lag} + beta_es * r_ES_{t-lag} + beta_dxy * r_DXY_{t-lag}
    """
    # Prepare lagged features
    features = pd.DataFrame({
        'const': 1.0,
        'brent_lag': returns_df['Brent'].shift(lag),
        'es_lag': returns_df['ES'].shift(lag),
        'dxy_lag': returns_df['DXY'].shift(lag),
    }, index=returns_df.index).dropna()
    
    target = returns_df['VWRA'].reindex(features.index)
    common = features.index.intersection(target.dropna().index)
    features = features.loc[common]
    target = target.loc[common]
    
    n_states = features.shape[1]
    n_obs = len(features)
    
    # Kalman filter initialization
    x = np.zeros(n_states)  # State estimate [alpha, beta_brent, beta_es, beta_dxy]
    P = np.eye(n_states) * 0.01  # State covariance
    Q = np.eye(n_states) * process_noise  # Process noise
    R = obs_noise  # Observation noise (scalar)
    
    # Storage
    betas = np.zeros((n_obs, n_states))
    predictions = np.zeros(n_obs)
    residuals = np.zeros(n_obs)
    
    X = features.values
    y = target.values
    
    for t in range(n_obs):
        H = X[t:t+1, :]  # 1 x n_states observation matrix
        
        # Predict
        x_pred = x  # State transition is identity
        P_pred = P + Q
        
        # Update
        y_pred = H @ x_pred
        residual = y[t] - y_pred[0]
        S = H @ P_pred @ H.T + R  # Innovation covariance
        K = P_pred @ H.T / S[0, 0]  # Kalman gain
        
        x = x_pred + K.flatten() * residual
        P = (np.eye(n_states) - K @ H) @ P_pred
        
        betas[t] = x
        predictions[t] = y_pred[0]
        residuals[t] = residual
    
    # Build results
    result = pd.DataFrame({
        'alpha': betas[:, 0],
        'beta_brent': betas[:, 1],
        'beta_es': betas[:, 2],
        'beta_dxy': betas[:, 3],
        'prediction': predictions,
        'residual': residuals,
        'actual': y,
    }, index=features.index)
    
    # Residual z-score for signal generation
    result['resid_std'] = result['residual'].rolling(48).std()
    result['resid_z'] = result['residual'] / result['resid_std']
    
    # Forward prediction: use current features to predict next VWRA move
    # Current (un-lagged) features as predictor for next period
    current_features = pd.DataFrame({
        'const': 1.0,
        'brent': returns_df['Brent'],
        'es': returns_df['ES'],
        'dxy': returns_df['DXY'],
    }, index=returns_df.index).reindex(result.index)
    
    result['fwd_prediction'] = (
        result['alpha'] + 
        result['beta_brent'] * current_features['brent'] +
        result['beta_es'] * current_features['es'] +
        result['beta_dxy'] * current_features['dxy']
    )
    
    # Signal: trade when predicted move exceeds cost threshold
    cost_frac = cost_threshold_bps / 10000
    result['signal'] = 0
    result.loc[result['fwd_prediction'] > cost_frac, 'signal'] = 1   # Buy
    result.loc[result['fwd_prediction'] < -cost_frac, 'signal'] = -1  # Sell
    result['strength'] = (result['fwd_prediction'].abs() / cost_frac).clip(0, 1)
    
    return result.dropna()

# Run Algo 3
algo3_signals = algo3_kalman_beta(returns, close_prices, lag=1)

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [2, 1.5, 1]})

# Dynamic betas over time
ax = axes[0]
for col, label, color in [('beta_brent', 'β Brent', '#e74c3c'), 
                            ('beta_es', 'β ES', '#3498db'),
                            ('beta_dxy', 'β DXY', '#f39c12')]:
    ax.plot(algo3_signals.index, algo3_signals[col], label=label, linewidth=1.5, color=color, alpha=0.8)
ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
ax.set_title('Algo 3: Kalman Filter Dynamic Betas (VWRA = α + β₁·Brent + β₂·ES + β₃·DXY)', fontsize=13)
ax.set_ylabel('Beta Coefficient')
ax.legend(fontsize=10)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Prediction vs actual
ax = axes[1]
ax.plot(algo3_signals.index, algo3_signals['actual'] * 10000, color='white', alpha=0.5, linewidth=0.8, label='Actual VWRA ret (bps)')
ax.plot(algo3_signals.index, algo3_signals['prediction'] * 10000, color='#2ecc71', linewidth=1.5, label='Predicted (bps)')
ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
ax.set_ylabel('Return (bps)')
ax.set_title('Model Prediction vs Actual VWRA Returns')
ax.legend(fontsize=9)

# Signal
ax = axes[2]
sig_colors = algo3_signals['signal'].map({-1: '#e74c3c', 0: '#95a5a6', 1: '#2ecc71'})
ax.bar(algo3_signals.index, algo3_signals['signal'], color=sig_colors, alpha=0.7, width=0.04)
ax.set_ylabel('Signal')
ax.set_title('Trading Signal (trade when |predicted move| > cost threshold)')
ax.set_ylim(-1.5, 1.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

# Model quality
r2 = 1 - (algo3_signals['residual']**2).sum() / ((algo3_signals['actual'] - algo3_signals['actual'].mean())**2).sum()
print(f"\nAlgo 3 — Kalman Filter Model:")
print(f"  In-sample R²: {r2:.4f}")
print(f"  Current betas — Brent: {algo3_signals['beta_brent'].iloc[-1]:.4f}, "
      f"ES: {algo3_signals['beta_es'].iloc[-1]:.4f}, DXY: {algo3_signals['beta_dxy'].iloc[-1]:.4f}")
print(f"  Forward prediction: {algo3_signals['fwd_prediction'].iloc[-1]*10000:.1f} bps")
print(f"  Current signal: {['SELL','FLAT','BUY'][algo3_signals['signal'].iloc[-1]+1]}")

## 7. Algorithm 4: Cross-Asset Confirmation Momentum

Trade VWRA only when the oil move is **confirmed** by other risk-off/risk-on assets. Filters out cases where oil moves for non-equity-relevant reasons.

In [ ]:
# ============================================================
# Cell 9: Algo 4 — Cross-Asset Confirmation Momentum
# ============================================================
# SELL VWRA when: Brent up + ES/NQ down + DXY/Gold up + VWRA hasn't repriced
# BUY VWRA when: Brent down + ES/NQ up + DXY/Gold down + VWRA hasn't repriced

def algo4_cross_asset_momentum(returns_df, prices_df, 
                                 oil_threshold=0.003,  # 0.3% min Brent move
                                 lookback=4):           # 4-hour lookback for momentum
    """
    Cross-asset confirmation momentum.
    Requires multiple assets to agree on direction before trading.
    """
    df = pd.DataFrame(index=returns_df.index)
    
    # Multi-hour returns for each asset
    for asset in ['Brent', 'ES', 'NQ', 'DXY', 'Gold', 'VWRA']:
        df[f'{asset}_mom'] = prices_df[asset].pct_change(lookback)
    
    # Confirmation scores: how many assets confirm risk-off vs risk-on?
    # Risk-off: oil up, equities down, DXY up, gold up
    df['oil_up'] = (df['Brent_mom'] > oil_threshold).astype(float)
    df['oil_down'] = (df['Brent_mom'] < -oil_threshold).astype(float)
    df['eq_down'] = ((df['ES_mom'] < 0) & (df['NQ_mom'] < 0)).astype(float)
    df['eq_up'] = ((df['ES_mom'] > 0) & (df['NQ_mom'] > 0)).astype(float)
    df['safe_haven_up'] = ((df['DXY_mom'] > 0) | (df['Gold_mom'] > 0)).astype(float)
    df['safe_haven_down'] = ((df['DXY_mom'] < 0) | (df['Gold_mom'] < 0)).astype(float)
    
    # VWRA hasn't fully repriced (lagging the move)
    # Compare VWRA move to what ES/NQ did — if VWRA moved less, there's room
    df['vwra_lag'] = df['VWRA_mom'].abs() < df[['ES_mom', 'NQ_mom']].abs().mean(axis=1) * 0.7
    
    # Risk-off confirmation count
    df['riskoff_confirms'] = df['oil_up'] + df['eq_down'] + df['safe_haven_up']
    df['riskon_confirms'] = df['oil_down'] + df['eq_up'] + df['safe_haven_down']
    
    # Signal: need at least 2 out of 3 confirmations + VWRA lagging
    df['signal'] = 0
    
    # SELL: strong risk-off confirmation
    sell_mask = (df['riskoff_confirms'] >= 2) & df['vwra_lag']
    df.loc[sell_mask, 'signal'] = -1
    
    # BUY: strong risk-on confirmation  
    buy_mask = (df['riskon_confirms'] >= 2) & df['vwra_lag']
    df.loc[buy_mask, 'signal'] = 1
    
    # Strength based on confirmation count and oil move size
    df['strength'] = 0.0
    df.loc[sell_mask, 'strength'] = (df.loc[sell_mask, 'riskoff_confirms'] / 3 * 
                                      (df.loc[sell_mask, 'Brent_mom'].abs() / oil_threshold).clip(1, 3) / 3)
    df.loc[buy_mask, 'strength'] = (df.loc[buy_mask, 'riskon_confirms'] / 3 *
                                     (df.loc[buy_mask, 'Brent_mom'].abs() / oil_threshold).clip(1, 3) / 3)
    df['strength'] = df['strength'].clip(0, 1)
    
    return df.dropna()

algo4_signals = algo4_cross_asset_momentum(returns, close_prices)

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [1, 1, 1]})

# Confirmation counts
ax = axes[0]
ax.fill_between(algo4_signals.index, algo4_signals['riskoff_confirms'], alpha=0.5, color='#e74c3c', label='Risk-Off Confirms')
ax.fill_between(algo4_signals.index, -algo4_signals['riskon_confirms'], alpha=0.5, color='#2ecc71', label='Risk-On Confirms')
ax.axhline(y=2, color='red', linestyle=':', alpha=0.5, label='Threshold (2)')
ax.axhline(y=-2, color='green', linestyle=':', alpha=0.5)
ax.set_title('Algo 4: Cross-Asset Confirmation Counts')
ax.set_ylabel('Confirmations')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Multi-asset momentum heatmap-style
ax = axes[1]
mom_cols = ['Brent_mom', 'ES_mom', 'NQ_mom', 'DXY_mom', 'Gold_mom', 'VWRA_mom']
for i, col in enumerate(mom_cols):
    name = col.replace('_mom', '')
    vals = algo4_signals[col].rolling(4).mean() * 100
    ax.plot(algo4_signals.index, vals + i * 2, label=name, linewidth=1.2)
    ax.axhline(y=i * 2, color='white', linewidth=0.3, alpha=0.3)
ax.set_title('4-Hour Rolling Momentum by Asset (%)')
ax.legend(fontsize=8, ncol=6, loc='upper center')

# Signal
ax = axes[2]
sig_colors = algo4_signals['signal'].map({-1: '#e74c3c', 0: '#95a5a6', 1: '#2ecc71'})
ax.bar(algo4_signals.index, algo4_signals['signal'], color=sig_colors, alpha=0.7, width=0.04)
ax.set_ylabel('Signal')
ax.set_title('Trading Signal')
ax.set_ylim(-1.5, 1.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

active = algo4_signals[algo4_signals['signal'] != 0]
print(f"\nAlgo 4 — Cross-Asset Confirmation:")
print(f"  Signal bars: {len(active)} / {len(algo4_signals)} ({len(active)/len(algo4_signals)*100:.1f}%)")
print(f"  Buy signals: {(active['signal']==1).sum()}, Sell signals: {(active['signal']==-1).sum()}")
print(f"  Current signal: {['SELL','FLAT','BUY'][algo4_signals['signal'].iloc[-1]+1]}")

## 8. Algorithm 5: Failed Oil-Spike Reversal

After a sharp oil spike (>2-3 sigma), if Brent retraces 40-60% quickly and VWRA still reflects the original risk-off move, trade the reversal in VWRA. Designed for the current headline-driven whipsaw environment.

In [ ]:
# ============================================================
# Cell 10: Algo 5 — Failed Oil-Spike Reversal
# ============================================================
# Detect large Brent spikes, then check if they reverse quickly.
# If VWRA hasn't caught up to the reversal → trade it.

def algo5_failed_spike(returns_df, prices_df, 
                        spike_sigma=2.0,       # Minimum spike size in sigma
                        retrace_min=0.30,      # Min retracement (30%)
                        retrace_max=0.70,      # Max retracement (70%)
                        spike_window=2,        # Hours for spike detection
                        retrace_window=4,      # Hours to observe retracement
                        signal_duration=4):    # Hours to hold signal
    """
    Failed oil-spike reversal model.
    """
    df = pd.DataFrame(index=returns_df.index)
    
    brent = prices_df['Brent']
    vwra = prices_df['VWRA']
    
    # Rolling statistics for spike detection
    brent_ret = returns_df['Brent']
    df['brent_ret'] = brent_ret
    df['brent_mean'] = brent_ret.rolling(48).mean()
    df['brent_std'] = brent_ret.rolling(48).std()
    df['brent_z'] = (brent_ret - df['brent_mean']) / df['brent_std']
    
    # Multi-bar spike: max |z-score| over spike_window
    df['spike_z'] = df['brent_z'].rolling(spike_window).apply(
        lambda x: x[np.argmax(np.abs(x))] if len(x) > 0 else 0, raw=True
    )
    
    # Detect spikes
    df['is_spike_up'] = df['spike_z'] > spike_sigma
    df['is_spike_down'] = df['spike_z'] < -spike_sigma
    
    # For each spike, compute retracement over the next retrace_window bars
    # Retracement = how much of the spike move has been given back
    df['brent_change_spike'] = brent.pct_change(spike_window)
    df['brent_change_after'] = brent.pct_change(retrace_window).shift(-retrace_window)
    
    # Retracement ratio: after_move / spike_move (negative = reversal)
    df['retrace_ratio'] = np.where(
        df['brent_change_spike'].abs() > 0.001,
        -df['brent_change_after'] / df['brent_change_spike'],
        0
    )
    
    # VWRA's response to the spike vs the reversal
    df['vwra_spike_move'] = vwra.pct_change(spike_window)
    df['vwra_after_move'] = vwra.pct_change(retrace_window).shift(-retrace_window)
    
    # Signal logic
    df['signal'] = 0
    df['strength'] = 0.0
    df['spike_type'] = 'NONE'
    
    # Upward spike that fails → BUY VWRA (oil spike scared equities, but it's reversing)
    up_spike_fail = (
        df['is_spike_up'] & 
        (df['retrace_ratio'] >= retrace_min) & 
        (df['retrace_ratio'] <= retrace_max)
    )
    df.loc[up_spike_fail, 'signal'] = 1  # Buy — the risk-off was overdone
    df.loc[up_spike_fail, 'spike_type'] = 'UP_SPIKE_FAIL'
    df.loc[up_spike_fail, 'strength'] = (df.loc[up_spike_fail, 'spike_z'].abs() / spike_sigma).clip(0.5, 1.0)
    
    # Downward spike that fails → SELL VWRA (relief rally was overdone)
    down_spike_fail = (
        df['is_spike_down'] &
        (df['retrace_ratio'] >= retrace_min) &
        (df['retrace_ratio'] <= retrace_max)
    )
    df.loc[down_spike_fail, 'signal'] = -1  # Sell — the relief was overdone
    df.loc[down_spike_fail, 'spike_type'] = 'DOWN_SPIKE_FAIL'
    df.loc[down_spike_fail, 'strength'] = (df.loc[down_spike_fail, 'spike_z'].abs() / spike_sigma).clip(0.5, 1.0)
    
    # Extend signals for signal_duration hours (hold the trade)
    signal_extended = df['signal'].copy()
    for i in range(1, signal_duration):
        signal_extended = signal_extended.where(signal_extended != 0, df['signal'].shift(i))
    df['signal'] = signal_extended.fillna(0).astype(int)
    
    return df.dropna()

algo5_signals = algo5_failed_spike(returns, close_prices)

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(18, 12), gridspec_kw={'height_ratios': [1.5, 1, 1]})

# Brent z-scores with spike markers
ax = axes[0]
ax.plot(algo5_signals.index, algo5_signals['brent_z'], color='#e67e22', linewidth=1, alpha=0.7, label='Brent Z-score')
ax.axhline(y=2, color='red', linestyle=':', alpha=0.5, label=f'{spike_sigma}σ threshold' if 'spike_sigma' in dir() else '2σ')
ax.axhline(y=-2, color='green', linestyle=':', alpha=0.5)
ax.axhline(y=0, color='white', linewidth=0.5, alpha=0.3)

# Mark failed spikes
for spike_type, marker, color in [('UP_SPIKE_FAIL', 'v', '#2ecc71'), ('DOWN_SPIKE_FAIL', '^', '#e74c3c')]:
    mask = algo5_signals['spike_type'] == spike_type
    if mask.any():
        ax.scatter(algo5_signals.index[mask], algo5_signals.loc[mask, 'brent_z'],
                  marker=marker, s=100, color=color, zorder=5, edgecolors='white',
                  label=f'{spike_type.replace("_", " ").title()}')

ax.set_title('Algo 5: Failed Oil-Spike Reversal — Brent Z-Scores & Spike Events')
ax.set_ylabel('Z-Score')
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Retracement ratios
ax = axes[1]
valid_retrace = algo5_signals[algo5_signals['spike_type'] != 'NONE']
if len(valid_retrace) > 0:
    colors = valid_retrace['spike_type'].map({'UP_SPIKE_FAIL': '#2ecc71', 'DOWN_SPIKE_FAIL': '#e74c3c'})
    ax.scatter(valid_retrace.index, valid_retrace['retrace_ratio'] * 100, c=colors, s=60, alpha=0.7, edgecolors='white')
ax.axhline(y=30, color='yellow', linestyle=':', alpha=0.5, label='30% retrace')
ax.axhline(y=70, color='yellow', linestyle=':', alpha=0.5, label='70% retrace')
ax.set_ylabel('Retracement %')
ax.set_title('Spike Retracement Ratios')
ax.legend(fontsize=9)

# Signal
ax = axes[2]
sig_colors = algo5_signals['signal'].map({-1: '#e74c3c', 0: '#95a5a6', 1: '#2ecc71'})
ax.bar(algo5_signals.index, algo5_signals['signal'], color=sig_colors, alpha=0.7, width=0.04)
ax.set_ylabel('Signal')
ax.set_title('Trading Signal')
ax.set_ylim(-1.5, 1.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

spikes = algo5_signals[algo5_signals['spike_type'] != 'NONE']
print(f"\nAlgo 5 — Failed Spike Reversal:")
print(f"  Total spikes detected: {len(spikes)}")
print(f"  Up-spike failures (→ BUY): {(spikes['spike_type']=='UP_SPIKE_FAIL').sum()}")
print(f"  Down-spike failures (→ SELL): {(spikes['spike_type']=='DOWN_SPIKE_FAIL').sum()}")
print(f"  Current signal: {['SELL','FLAT','BUY'][algo5_signals['signal'].iloc[-1]+1]}")

## 9. Algorithm 6: Fair-Value Residual / Z-Score Model

Build a real-time "fair VWRA" from Brent, ES/NQ, DXY, and VWRA's own lagged return. Trade when actual VWRA deviates from model fair value by a large z-score. Says "VWRA is too cheap/rich relative to the current macro tape."

In [ ]:
# ============================================================
# Cell 11: Algo 6 — Fair-Value Residual / Z-Score Model
# ============================================================
# Rolling OLS: VWRA ~ Brent + ES + DXY + Gold + VWRA_lag
# Trade the residual when |z| > threshold

def algo6_fair_value_zscore(returns_df, prices_df, 
                             fit_window=72,    # 72-hour rolling regression window
                             z_threshold=1.5,  # Z-score threshold for signal
                             z_exit=0.5):      # Z-score for exit
    """
    Fair-value residual model with rolling OLS regression.
    """
    df = pd.DataFrame(index=returns_df.index)
    
    # Features: contemporaneous returns of macro drivers + lagged VWRA
    df['y'] = returns_df['VWRA']
    df['brent'] = returns_df['Brent']
    df['es'] = returns_df['ES']
    df['dxy'] = returns_df['DXY']
    df['gold'] = returns_df['Gold']
    df['vwra_lag1'] = returns_df['VWRA'].shift(1)
    df['vwra_lag2'] = returns_df['VWRA'].shift(2)
    df = df.dropna()
    
    feature_cols = ['brent', 'es', 'dxy', 'gold', 'vwra_lag1', 'vwra_lag2']
    
    # Rolling regression
    predictions = []
    residuals = []
    r2s = []
    
    for i in range(fit_window, len(df)):
        window = df.iloc[i - fit_window:i]
        X = window[feature_cols].values
        y = window['y'].values
        
        # Add constant
        X_c = np.column_stack([np.ones(len(X)), X])
        
        # OLS: beta = (X'X)^-1 X'y
        try:
            beta = np.linalg.lstsq(X_c, y, rcond=None)[0]
            
            # Predict current point
            x_now = np.concatenate([[1], df.iloc[i][feature_cols].values])
            pred = x_now @ beta
            resid = df.iloc[i]['y'] - pred
            
            # In-sample R2
            y_hat = X_c @ beta
            ss_res = ((y - y_hat)**2).sum()
            ss_tot = ((y - y.mean())**2).sum()
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        except:
            pred = 0
            resid = 0
            r2 = 0
        
        predictions.append(pred)
        residuals.append(resid)
        r2s.append(r2)
    
    result = df.iloc[fit_window:].copy()
    result['fair_value_ret'] = predictions
    result['residual'] = residuals
    result['rolling_r2'] = r2s
    
    # Z-score of residual
    result['resid_mean'] = result['residual'].rolling(48).mean()
    result['resid_std'] = result['residual'].rolling(48).std()
    result['z_score'] = (result['residual'] - result['resid_mean']) / result['resid_std']
    
    # Fair value in price terms
    # Cumulate fair-value returns to get fair price level
    vwra_prices = prices_df['VWRA'].reindex(result.index)
    result['vwra_price'] = vwra_prices
    result['fair_price_cum_resid'] = result['residual'].cumsum()
    
    # Signal: mean-reversion on residual
    # Positive residual = VWRA is ABOVE fair value → SELL
    # Negative residual = VWRA is BELOW fair value → BUY
    result['signal'] = 0
    result.loc[result['z_score'] < -z_threshold, 'signal'] = 1   # Below fair value → BUY
    result.loc[result['z_score'] > z_threshold, 'signal'] = -1   # Above fair value → SELL
    
    # Hold signal until z-score crosses back through exit threshold
    # Simple implementation: carry forward signal
    current_signal = 0
    signals = []
    for _, row in result.iterrows():
        if row['z_score'] < -z_threshold:
            current_signal = 1
        elif row['z_score'] > z_threshold:
            current_signal = -1
        elif abs(row['z_score']) < z_exit:
            current_signal = 0
        signals.append(current_signal)
    result['signal'] = signals
    
    result['strength'] = (result['z_score'].abs() / z_threshold).clip(0, 1)
    
    return result.dropna()

algo6_signals = algo6_fair_value_zscore(returns, close_prices)

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [1.5, 1.5, 1]})

# Z-score with buy/sell zones
ax = axes[0]
ax.plot(algo6_signals.index, algo6_signals['z_score'], color='#3498db', linewidth=1.5, label='Residual Z-Score')
ax.fill_between(algo6_signals.index, 1.5, 4, alpha=0.15, color='red', label='Sell Zone (z > 1.5)')
ax.fill_between(algo6_signals.index, -4, -1.5, alpha=0.15, color='green', label='Buy Zone (z < -1.5)')
ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
ax.set_title('Algo 6: Fair-Value Residual Z-Score — VWRA vs Macro Fair Value', fontsize=13)
ax.set_ylabel('Z-Score')
ax.set_ylim(-4, 4)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Rolling R2 of fair-value model
ax = axes[1]
ax.plot(algo6_signals.index, algo6_signals['rolling_r2'], color='#f39c12', linewidth=1.5)
ax.axhline(y=algo6_signals['rolling_r2'].mean(), color='cyan', linestyle=':', 
           label=f"Mean R²={algo6_signals['rolling_r2'].mean():.3f}")
ax.set_title('Rolling 72-Hour Model R² (how well macro explains VWRA)')
ax.set_ylabel('R²')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

# Signal
ax = axes[2]
sig_colors_list = ['#e74c3c' if s == -1 else '#2ecc71' if s == 1 else '#95a5a6' for s in algo6_signals['signal']]
ax.bar(algo6_signals.index, algo6_signals['signal'], color=sig_colors_list, alpha=0.7, width=0.04)
ax.set_ylabel('Signal')
ax.set_title('Trading Signal (mean-reversion on fair-value residual)')
ax.set_ylim(-1.5, 1.5)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.show()

print(f"\nAlgo 6 — Fair-Value Z-Score:")
print(f"  Rolling R² — Mean: {algo6_signals['rolling_r2'].mean():.4f}, Std: {algo6_signals['rolling_r2'].std():.4f}")
print(f"  Current Z-score: {algo6_signals['z_score'].iloc[-1]:.2f}")
print(f"  Current signal: {['SELL','FLAT','BUY'][algo6_signals['signal'].iloc[-1]+1]}")
print(f"  Interpretation: VWRA is {'CHEAP' if algo6_signals['z_score'].iloc[-1] < 0 else 'RICH'} "
      f"relative to macro fair value by {abs(algo6_signals['z_score'].iloc[-1]):.1f}σ")

## 10. Backtest Comparison — All Algorithms

Run all 6 algos through the backtest engine with IBKR cost model. Compare performance across 1-hour and 4-hour horizons.

In [ ]:
# ============================================================
# Cell 12: Backtest All Algorithms
# ============================================================

# Align all signal DataFrames to common VWRA price series
vwra_prices = close_prices['VWRA'].dropna()

all_algos = {
    '1. Regime Switch': algo1_signals,
    '2. London Gap': algo2_signals,
    '3. Kalman Beta': algo3_signals,
    '4. Cross-Asset': algo4_signals,
    '5. Spike Reversal': algo5_signals,
    '6. Fair Value Z': algo6_signals,
}

# Run backtests for both horizons
all_results = {}
summary_rows = []

for horizon in [1, 4]:
    for name, sig_df in all_algos.items():
        # Align signals with VWRA prices
        common_idx = sig_df.index.intersection(vwra_prices.index)
        if len(common_idx) < 10:
            continue
        
        aligned_signals = sig_df.loc[common_idx][['signal', 'strength']].copy()
        aligned_prices = vwra_prices.loc[common_idx]
        
        bt = backtest_signals(aligned_signals, aligned_prices, horizon_hours=horizon)
        key = f"{name} ({horizon}h)"
        all_results[key] = bt
        
        stats = summarize_backtest(bt, name=key)
        summary_rows.append(stats)

summary_df = pd.DataFrame(summary_rows)
print("=" * 100)
print("BACKTEST RESULTS — All Algorithms (with IBKR cost model)")
print("=" * 100)
display(summary_df.set_index('Strategy'))

# Cumulative P&L chart
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for h_idx, horizon in enumerate([1, 4]):
    ax = axes[h_idx]
    for name in all_algos.keys():
        key = f"{name} ({horizon}h)"
        if key in all_results:
            bt = all_results[key]
            ax.plot(bt.index, bt['cum_pnl'] * 10000, linewidth=2, label=name, alpha=0.8)
    
    ax.axhline(y=0, color='white', linewidth=0.5, linestyle='--')
    ax.set_title(f'Cumulative Net P&L — {horizon}-Hour Horizon', fontsize=13)
    ax.set_ylabel('Cumulative P&L (bps)')
    ax.legend(fontsize=9, loc='best')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))

plt.tight_layout()
plt.suptitle('Backtest: All Algorithms — Net of IBKR Costs', fontsize=14, fontweight='bold', y=1.02)
plt.show()

# Win rate comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
for h_idx, horizon in enumerate([1, 4]):
    ax = axes[h_idx]
    names = []
    total_rets = []
    colors = []
    for name in all_algos.keys():
        key = f"{name} ({horizon}h)"
        if key in all_results:
            bt = all_results[key]
            tr = bt['cum_pnl'].iloc[-1] * 10000
            total_rets.append(tr)
            names.append(name)
            colors.append('#2ecc71' if tr > 0 else '#e74c3c')
    
    bars = ax.barh(names, total_rets, color=colors, alpha=0.8, edgecolor='white')
    ax.axvline(x=0, color='white', linewidth=0.5)
    ax.set_xlabel('Total Net Return (bps)')
    ax.set_title(f'{horizon}-Hour Horizon')
    
    for bar, val in zip(bars, total_rets):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, 
                f'{val:.1f}', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.suptitle('Total Net Return by Algorithm', fontsize=14, fontweight='bold', y=1.02)
plt.show()

## 11. Interactive Trading Dashboard

**Scrollable dashboard** showing all algo signals over time. Use the range slider to scroll through any hour in the past month. The "NOW" panel shows current live signals.

In [ ]:
# ============================================================
# Cell 13: Current Signal Summary — "WHAT TO DO RIGHT NOW"
# ============================================================
from datetime import datetime

print("=" * 80)
print(f"  LIVE SIGNAL DASHBOARD — {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print("=" * 80)

signal_map = {-1: '🔴 SELL', 0: '⚪ FLAT', 1: '🟢 BUY'}
algo_names = {
    '1. Regime Switch': algo1_signals,
    '2. London Gap': algo2_signals,
    '3. Kalman Beta': algo3_signals,
    '4. Cross-Asset': algo4_signals, 
    '5. Spike Reversal': algo5_signals,
    '6. Fair Value Z': algo6_signals,
}

votes = {'BUY': 0, 'SELL': 0, 'FLAT': 0}
print(f"\n{'Algorithm':<25s} {'Signal':<12s} {'Strength':>10s}  Details")
print("-" * 80)

for name, sig_df in algo_names.items():
    last = sig_df.iloc[-1]
    sig = int(last['signal'])
    strength = last.get('strength', 0)
    sig_text = signal_map[sig]
    
    # Extra details per algo
    detail = ""
    if '1.' in name and 'regime' in last.index:
        detail = f"Regime: {last['regime']}"
    elif '2.' in name and 'variant' in last.index:
        detail = f"Variant: {last['variant']}"
    elif '3.' in name and 'fwd_prediction' in last.index:
        detail = f"Predicted: {last['fwd_prediction']*10000:.1f}bps"
    elif '4.' in name and 'riskoff_confirms' in last.index:
        detail = f"Risk-off: {last['riskoff_confirms']:.0f}/3, Risk-on: {last['riskon_confirms']:.0f}/3"
    elif '5.' in name and 'brent_z' in last.index:
        detail = f"Brent Z: {last['brent_z']:.1f}σ"
    elif '6.' in name and 'z_score' in last.index:
        detail = f"Fair-value Z: {last['z_score']:.2f}σ"
    
    print(f"  {name:<23s} {sig_text:<12s} {strength:>8.1%}   {detail}")
    
    if sig == 1: votes['BUY'] += 1
    elif sig == -1: votes['SELL'] += 1
    else: votes['FLAT'] += 1

# Consensus
print(f"\n{'='*80}")
print(f"  CONSENSUS: BUY={votes['BUY']}  SELL={votes['SELL']}  FLAT={votes['FLAT']}")
if votes['BUY'] > votes['SELL'] and votes['BUY'] >= 2:
    consensus = "🟢 LEAN BUY"
elif votes['SELL'] > votes['BUY'] and votes['SELL'] >= 2:
    consensus = "🔴 LEAN SELL"  
else:
    consensus = "⚪ NO CLEAR SIGNAL"
print(f"  OVERALL: {consensus}")
print(f"{'='*80}")

# Key market context
print(f"\n  Market Context:")
print(f"    VWRA:  {close_prices['VWRA'].iloc[-1]:.2f} GBP")
print(f"    Brent: ${close_prices['Brent'].iloc[-1]:.2f}")
print(f"    WTI:   ${close_prices['WTI'].iloc[-1]:.2f}")
print(f"    Spread: ${close_prices['Brent_WTI_spread'].iloc[-1]:.2f}")
print(f"    ES:    {close_prices['ES'].iloc[-1]:.0f}")
print(f"    DXY:   {close_prices['DXY'].iloc[-1]:.2f}")
print(f"    Gold:  ${close_prices['Gold'].iloc[-1]:.0f}")

In [ ]:
# ============================================================
# Cell 14: Interactive Plotly Dashboard — Scrollable History
# ============================================================
# Full interactive dashboard with range slider to scroll through
# any hour in the past month and see what each algo said.

# Build unified signal matrix
signal_matrix = pd.DataFrame(index=close_prices.index)
for name, sig_df in all_algos.items():
    signal_matrix[name] = sig_df['signal'].reindex(signal_matrix.index).fillna(0)

# Consensus
signal_matrix['Consensus'] = signal_matrix[list(all_algos.keys())].sum(axis=1)
signal_matrix['VWRA'] = close_prices['VWRA']
signal_matrix['Brent'] = close_prices['Brent']
signal_matrix = signal_matrix.dropna(subset=['VWRA', 'Brent'])

# Create the mega-dashboard
fig = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    row_heights=[0.35, 0.25, 0.25, 0.15],
    subplot_titles=[
        'VWRA Price & Brent Crude',
        'Individual Algorithm Signals (hover for details)',
        'Consensus Score (sum of all signals, -6 to +6)',
        'Algo Signal Heatmap'
    ]
)

# Row 1: Price charts
fig.add_trace(go.Scatter(
    x=signal_matrix.index, y=signal_matrix['VWRA'],
    name='VWRA (GBP)', line=dict(color='#00d4ff', width=2),
    yaxis='y1'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=signal_matrix.index, y=signal_matrix['Brent'],
    name='Brent ($)', line=dict(color='#ff6b35', width=2),
    yaxis='y2'
), row=1, col=1)

# Row 2: Individual algo signals as colored markers
algo_colors = {
    '1. Regime Switch': '#e74c3c',
    '2. London Gap': '#3498db', 
    '3. Kalman Beta': '#2ecc71',
    '4. Cross-Asset': '#f39c12',
    '5. Spike Reversal': '#9b59b6',
    '6. Fair Value Z': '#1abc9c',
}

for i, (name, color) in enumerate(algo_colors.items()):
    sig = signal_matrix[name]
    
    # Buy markers
    buy_mask = sig == 1
    if buy_mask.any():
        fig.add_trace(go.Scatter(
            x=signal_matrix.index[buy_mask], y=[i] * buy_mask.sum(),
            mode='markers', marker=dict(symbol='triangle-up', size=8, color='#00ff88'),
            name=f'{name} BUY', showlegend=False,
            hovertemplate=f'{name}<br>BUY<br>%{{x}}<extra></extra>'
        ), row=2, col=1)
    
    # Sell markers
    sell_mask = sig == -1
    if sell_mask.any():
        fig.add_trace(go.Scatter(
            x=signal_matrix.index[sell_mask], y=[i] * sell_mask.sum(),
            mode='markers', marker=dict(symbol='triangle-down', size=8, color='#ff4444'),
            name=f'{name} SELL', showlegend=False,
            hovertemplate=f'{name}<br>SELL<br>%{{x}}<extra></extra>'
        ), row=2, col=1)

# Row 3: Consensus
consensus = signal_matrix['Consensus']
consensus_colors = ['#00ff88' if v > 0 else '#ff4444' if v < 0 else '#666666' for v in consensus]

fig.add_trace(go.Bar(
    x=signal_matrix.index, y=consensus,
    marker_color=consensus_colors,
    name='Consensus', showlegend=False,
    hovertemplate='Consensus: %{y}<br>%{x}<extra></extra>'
), row=3, col=1)

# Add threshold lines
fig.add_hline(y=2, line_dash="dot", line_color="green", opacity=0.5, row=3, col=1)
fig.add_hline(y=-2, line_dash="dot", line_color="red", opacity=0.5, row=3, col=1)
fig.add_hline(y=0, line_dash="solid", line_color="white", opacity=0.3, row=3, col=1)

# Row 4: Signal heatmap using colored bars for each algo
for i, (name, color) in enumerate(algo_colors.items()):
    sig = signal_matrix[name]
    # Create a trace where color encodes signal
    sig_colors_row = ['rgba(0,255,136,0.8)' if s == 1 else 'rgba(255,68,68,0.8)' if s == -1 else 'rgba(50,50,50,0.3)' for s in sig]
    fig.add_trace(go.Bar(
        x=signal_matrix.index, y=[1] * len(sig),
        base=[i] * len(sig),
        marker_color=sig_colors_row,
        showlegend=False,
        hovertemplate=f'{name}: ' + '%{text}<br>%{x}<extra></extra>',
        text=['BUY' if s == 1 else 'SELL' if s == -1 else 'FLAT' for s in sig],
    ), row=4, col=1)

# Layout
fig.update_layout(
    height=1200,
    template='plotly_dark',
    title=dict(
        text='Oil-VWRA Trading Signal Dashboard — Scroll Through History',
        font=dict(size=18)
    ),
    hovermode='x unified',
    
    # Range slider on bottom x-axis
    xaxis4=dict(
        rangeslider=dict(visible=True, thickness=0.05),
        type='date',
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1d", step="day", stepmode="backward"),
                dict(count=3, label="3d", step="day", stepmode="backward"),
                dict(count=7, label="1w", step="day", stepmode="backward"),
                dict(count=14, label="2w", step="day", stepmode="backward"),
                dict(step="all", label="All"),
            ]),
            bgcolor='#1a1a2e',
            activecolor='#e94560',
        ),
    ),
    
    bargap=0,
    barmode='stack',
)

# Y-axis labels
fig.update_yaxes(title_text='Price', row=1, col=1)
fig.update_yaxes(
    title_text='Algo', row=2, col=1,
    tickvals=list(range(6)),
    ticktext=[n.split('. ')[1] for n in algo_colors.keys()],
    range=[-0.5, 5.5]
)
fig.update_yaxes(title_text='Score', range=[-6.5, 6.5], row=3, col=1)
fig.update_yaxes(
    title_text='', row=4, col=1,
    tickvals=[i + 0.5 for i in range(6)],
    ticktext=[n.split('. ')[1] for n in algo_colors.keys()],
    range=[0, 6]
)

# Add secondary y-axis for Brent on row 1
fig.update_layout(
    yaxis2=dict(
        title='Brent ($)',
        overlaying='y',
        side='right',
        showgrid=False,
    )
)

fig.show()

## 12. Summary & Interpretation

### How to read this notebook:
1. **Re-run all cells** to refresh with latest market data (Yahoo gives ~1 month of hourly history)
2. **Cell 13** gives you the instant "what to do right now" answer
3. **Cell 14** (Plotly dashboard) lets you scroll back through any hour and see what each algo would have said
4. The **backtest table** (Cell 12) shows which algos are actually making money net of IBKR costs

### Algorithm priority (per the spec):
- **Build first**: Algos 1 (Regime Switch), 2 (London Gap), 3 (Kalman Beta) — best fit for current war-driven regime and human execution speed
- **Confirmation layer**: Algo 4 (Cross-Asset) — filters noise
- **Opportunistic**: Algos 5 (Spike Reversal), 6 (Fair Value Z) — for specific market conditions

### Key caveats:
- This is a **temporary macro regime** — all models will degrade when the war-driven oil shock ends
- Only ~1 month of hourly data available from Yahoo; longer backtests need a paid data source
- The backtest is **in-sample** for the Kalman filter and rolling regression — overfitting is likely
- Walk-forward validation and out-of-sample testing would be needed before live trading
- VWRA is an ETF (not a future) — short-selling constraints and borrowing costs apply to SELL signals